In [10]:
library(data.table)
library(dplyr)
library(openxlsx)

# Output summary .xlsx files

In [11]:
dirs <- list.dirs("./results", recursive = FALSE)
out_dir <- file.path(".", "supplementary_tables")
dir.create(out_dir, showWarnings = FALSE)

for (dir_path in dirs) {
    rds_files <- list.files(dir_path, pattern = "\\.rds$", full.names = TRUE)

    for (rds_file in rds_files) {
        data <- readRDS(rds_file)

        summary_df <- data %>%
            dplyr::group_by(pathway) %>%
            dplyr::summarise(
                mean_padj = mean(padj[padj < 0.05], na.rm = TRUE),
                N         = sum(padj < 0.05, na.rm = TRUE),
                N_pos     = sum(padj < 0.05 & ES > 0, na.rm = TRUE),
                N_neg     = sum(padj < 0.05 & ES < 0, na.rm = TRUE),
                size      = mean(size),
                .groups   = "drop"
            ) %>%
            arrange(mean_padj)

        wb <- createWorkbook()
        addWorksheet(wb, "summary")
        writeData(wb, "summary", summary_df)

        out_file <- file.path(out_dir, paste0(tools::file_path_sans_ext(basename(rds_file)), ".xlsx"))
        saveWorkbook(wb, out_file, overwrite = TRUE)
    }
}

# View summary tables in notebook

In [14]:
## Cancer groups files
# data <- readRDS("./results/per_sample_gsea/normalized_Basenji_cancer_per_sample_GSEA.rds") 
# data <- readRDS("./results/per_sample_gsea/normalized_Enformer_cancer_per_sample_GSEA.rds") 

## Tissue groups files
# data <- readRDS("./results/per_sample_gsea/normalized_Basenji_tissue_per_sample_GSEA.rds") 
# data <- readRDS("./results/per_sample_gsea/normalized_Enformer_tissue_per_sample_GSEA.rds")

## Tissue & cancer groups files
data <- readRDS("./results/per_sample_gsea/normalized_Basenji_tissue_and_cancer_per_sample_GSEA.rds") 
# data <- readRDS("./results/per_sample_gsea/normalized_Enformer_tissue_and_cancer_per_sample_GSEA.rds") 

## Cancer differential files
# data <- readRDS("./results/differential_gsea/normalized_Basenji_cancer_differential_GSEA.rds") 
# data <- readRDS("./results/differential_gsea/normalized_Enformer_cancer_differential_GSEA.rds") 


In [15]:
data %>%
    dplyr::group_by(pathway) %>%
    dplyr::summarise(
    mean_padj = mean(padj[padj<0.05]),
    N     = sum(padj < 0.05, na.rm = TRUE),
    N_pos = sum(padj < 0.05 & ES > 0, na.rm = TRUE),
    N_neg = sum(padj < 0.05 & ES < 0, na.rm = TRUE),
    size = mean(size),
    ) %>%arrange(mean_padj)

pathway,mean_padj,N,N_pos,N_neg,size
<chr>,<dbl>,<int>,<int>,<int>,<dbl>
GOMF_OLFACTORY_RECEPTOR_ACTIVITY,4.532811e-08,72,0,72,43
GOBP_SENSORY_PERCEPTION_OF_SMELL,1.936805e-07,72,0,72,46
GOBP_SENSORY_PERCEPTION_OF_CHEMICAL_STIMULUS,2.914401e-07,72,0,72,61
GOBP_DETECTION_OF_STIMULUS_INVOLVED_IN_SENSORY_PERCEPTION,1.749323e-06,72,0,72,60
GOBP_DETECTION_OF_CHEMICAL_STIMULUS,4.364452e-05,72,0,72,61
GOBP_DETECTION_OF_STIMULUS,1.885765e-04,71,0,71,69
GOBP_SENSORY_PERCEPTION,2.661660e-04,72,0,72,93
GOBP_CELL_MORPHOGENESIS_INVOLVED_IN_NEURON_DIFFERENTIATION,4.814151e-04,1,1,0,57
GOBP_AXON_DEVELOPMENT,4.942061e-04,1,1,0,51


# Code graveyard

In [9]:
data <- data[order(data$padj),]
data <- data[padj < 0.01,.N,by=pathway,]
data[order(N,decreasing=TRUE)]

pathway,N
<chr>,<int>
GOMF_OLFACTORY_RECEPTOR_ACTIVITY,43
GOBP_SENSORY_PERCEPTION_OF_SMELL,43
GOBP_DETECTION_OF_STIMULUS_INVOLVED_IN_SENSORY_PERCEPTION,43
GOBP_SENSORY_PERCEPTION_OF_CHEMICAL_STIMULUS,43
GOMF_G_PROTEIN_COUPLED_RECEPTOR_ACTIVITY,43
GOBP_DETECTION_OF_CHEMICAL_STIMULUS,43
GOBP_DETECTION_OF_STIMULUS,43
GOBP_SENSORY_PERCEPTION,43
GOBP_G_PROTEIN_COUPLED_RECEPTOR_SIGNALING_PATHWAY,42


In [26]:
data[order(data$padj),]#[540:570,]

pathway,pval,padj,log2err,ES,NES,size,tissue
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<chr>
GOBP_DETECTION_OF_STIMULUS_INVOLVED_IN_SENSORY_PERCEPTION,9.874083e-37,2.796340e-33,1.576374,-0.8538032,-3.981657,60,bone_cancer
GOMF_G_PROTEIN_COUPLED_RECEPTOR_ACTIVITY,1.006634e-36,2.850789e-33,1.576374,-0.7867751,-3.875139,92,endocrine_cancer
GOMF_OLFACTORY_RECEPTOR_ACTIVITY,1.639924e-36,4.644264e-33,1.569792,-0.9257828,-3.733409,43,skin_cancer
GOBP_DETECTION_OF_STIMULUS_INVOLVED_IN_SENSORY_PERCEPTION,1.976581e-36,5.597678e-33,1.569792,-0.8582550,-4.136358,60,brain_noncancer
GOBP_DETECTION_OF_STIMULUS_INVOLVED_IN_SENSORY_PERCEPTION,2.200809e-36,6.232691e-33,1.569792,-0.8462993,-4.179724,60,kidney_noncancer
GOMF_OLFACTORY_RECEPTOR_ACTIVITY,4.756896e-36,6.735764e-33,1.563182,-0.9133951,-4.166949,43,kidney_noncancer
GOMF_OLFACTORY_RECEPTOR_ACTIVITY,3.520811e-36,9.970936e-33,1.563182,-0.9171595,-3.975910,43,male rep._cancer
GOBP_DETECTION_OF_STIMULUS_INVOLVED_IN_SENSORY_PERCEPTION,1.066257e-35,1.509820e-32,1.556544,-0.8618981,-3.826562,60,skin_cancer
GOBP_SENSORY_PERCEPTION_OF_SMELL,7.779897e-35,7.344223e-32,1.536459,-0.8936829,-4.147020,46,kidney_noncancer


In [63]:
data[data$pathway == 'GOMF_DNA_BINDING_TRANSCRIPTION_FACTOR_ACTIVITY']

pathway,pval,padj,log2err,ES,NES,size,tissue
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<chr>
GOMF_DNA_BINDING_TRANSCRIPTION_FACTOR_ACTIVITY,9.737219e-07,0.0001723488,0.6435518,0.3874525,1.878370,267,Cancer
GOMF_DNA_BINDING_TRANSCRIPTION_FACTOR_ACTIVITY,2.425638e-03,0.1248983044,0.4317077,0.3163796,1.526201,267,Not Cancer


In [ ]:
targets <- fread('~/basenji_human_data/targets.txt',
                        header = TRUE,
                       sep = '\t')
CAGE_cols <- targets[grepl('CAGE',targets$description),]
CAGE_cols$cage_indx <- 1:nrow(CAGE_cols)

In [ ]:
data <- readRDS("Basenji_median_GSEA.rds")

In [ ]:
data

In [ ]:
  data <- readRDS("Enformer_median_GSEA.rds")

In [ ]:
data

In [ ]:
data <- readRDS("Basenji_per_sample_GSEA.rds")
data <- data[order(data$padj),]
data$description <- CAGE_cols$description[ match(data$col_indx, CAGE_cols$cage_indx)]
saveRDS(data, file="Basenji_per_sample_GSEA_with_track_description.rds")

In [ ]:
data[data$pathway == 'GOMF_DNA_BINDING_TRANSCRIPTION_FACTOR_ACTIVITY']

In [ ]:
data <- readRDS("Enformer_per_sample_GSEA.rds")
data <- data[order(data$padj),]
data$description <- CAGE_cols$description[ match(data$col_indx, CAGE_cols$cage_indx)]
saveRDS(data, file="Enformer_per_sample_GSEA_with_track_description.rds")

In [ ]:
data